### Comparing SPIs: where the choice of statistic changes the answer

Running 328 statistics is only worth it if they disagree. They do, and predictably so: a
statistic can only detect the kind of structure it is built to detect.

This notebook builds a small dataset whose couplings we know exactly, then shows two
concrete failures of the default choice:

1. **linear statistics miss nonlinear coupling** -- correlation is ~0 between two
   processes that are deterministically related;
2. **undirected statistics cannot see direction** -- and contemporaneous ones miss a
   purely lagged coupling altogether.

> Plotting cells need matplotlib, which is not a core pyspi dependency:
> `pip install matplotlib`, or `pip install "pyspi[bench]"`.

In [ ]:
import os
import tempfile
import time
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pyspi.calculator import Calculator

RNG = np.random.default_rng(42)
T = 1000  # long enough that the effects below are real, not seed artefacts

#### 1. A dataset with known structure

Six processes, `T = 1000` observations:

| process  | construction                                | relationship |
|----------|---------------------------------------------|--------------|
| `drive`  | i.i.d. standard normal                      | -- |
| `lin`    | `0.8 * drive + noise`                       | **linear**, contemporaneous, with `drive` |
| `nonlin` | `drive**2 + noise`                          | **nonlinear**, contemporaneous, with `drive` |
| `src`    | i.i.d. standard normal                      | -- |
| `tgt`    | `0.85 * src[t-1] + noise`                   | **directed and lagged**, `src` -> `tgt` |
| `indep`  | i.i.d. standard normal                      | none |

Two properties are doing the work. `drive` is symmetric about zero, so
`corr(drive, drive**2) = 0` even though `nonlin` is *determined* by `drive` up to noise.
And `tgt` depends on `src` only at lag 1, so their contemporaneous correlation is also 0.

In [ ]:
drive = RNG.standard_normal(T)
lin = 0.8 * drive + 0.6 * RNG.standard_normal(T)
nonlin = drive ** 2 + 0.5 * RNG.standard_normal(T)

src = RNG.standard_normal(T)
noise = RNG.standard_normal(T)
tgt = np.empty(T)
tgt[0] = noise[0]
for t in range(1, T):
    tgt[t] = 0.85 * src[t - 1] + 0.5 * noise[t]

indep = RNG.standard_normal(T)

NAMES = ["drive", "lin", "nonlin", "src", "tgt", "indep"]
X = np.vstack([drive, lin, nonlin, src, tgt, indep])

print(f"pearson(drive, lin)    = {np.corrcoef(drive, lin)[0, 1]: .3f}")
print(f"pearson(drive, nonlin) = {np.corrcoef(drive, nonlin)[0, 1]: .3f}   <- near zero, but not independent")
print(f"pearson(src, tgt)      = {np.corrcoef(src, tgt)[0, 1]: .3f}   <- near zero, but src drives tgt at lag 1")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.4))

for ax, (x, y, title, xlab, ylab) in zip(axes, [
    (drive, lin, "linear: lin vs drive", "drive", "lin"),
    (drive, nonlin, "nonlinear: nonlin vs drive", "drive", "nonlin"),
    (src[:-1], tgt[1:], "lagged: tgt[t] vs src[t-1]", "src[t-1]", "tgt[t]"),
]):
    ax.scatter(x, y, s=5, alpha=0.35, color="#3b6ea5", edgecolors="none")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlab)
    ax.set_ylabel(ylab)
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(alpha=0.15)

fig.tight_layout()
plt.show()

#### 2. A hand-written config

The bundled configs are convenient but broad. For a targeted comparison it is clearer to
write the handful of SPIs we want. A config is YAML: module, then SPI class, then one
entry per parameter set.

(`pyspi.utils.filter_spis(["directed", "nonlinear"])` does the same thing by keyword
filtering of the full config, if you would rather not write YAML by hand.)

In [ ]:
CONFIG = """
.statistics.basic:
  Covariance:
    configs:
      - estimator: EmpiricalCovariance
  SpearmanR:
    configs:
      - squared: False

.statistics.distance:
  DistanceCorrelation:
    configs:
      - biased: False

.statistics.infotheory:
  MutualInfo:
    configs:
      - estimator: kraskov
        prop_k: 4
  TimeLaggedMutualInfo:
    configs:
      - estimator: gaussian
  TransferEntropy:
    configs:
      - estimator: kraskov
        prop_k: 4
        k_history: 1
        l_history: 1
"""

config_path = os.path.join(tempfile.mkdtemp(), "demo_config.yaml")
with open(config_path, "w") as f:
    f.write(CONFIG)

t0 = time.perf_counter()
calc = Calculator(dataset=X, config=config_path, verbose=False)
calc.compute(progress=False)
print(f"{calc.n_spis} SPIs in {time.perf_counter() - t0:.2f}s")

Six SPIs, split by what they can represent:

| identifier | undirected/directed | linear/nonlinear |
|---|---|---|
| `cov_EmpiricalCovariance` | undirected | linear |
| `spearmanr` | undirected | monotonic only |
| `dcorr` | undirected | nonlinear |
| `mi_kraskov_NN-4` | undirected | nonlinear |
| `tlmi_gaussian` | directed | linear (Gaussian estimator) |
| `te_kraskov_NN-4_k-1_kt-1_l-1_lt-1` | directed | nonlinear |

The `Data` object does not carry our process names through, so we relabel the matrices
by hand.

In [ ]:
def matrix(spi):
    m = calc.table[spi].copy()
    m.index = NAMES
    m.columns = NAMES
    return m

matrix("cov_EmpiricalCovariance").round(3)

#### 3. Linear statistics miss the nonlinear coupling

We compare four undirected SPIs on three pairs: the linear coupling, the nonlinear
coupling, and an uncoupled pair as a control.

Their units are not comparable to each other -- covariance is a correlation, mutual
information is in nats -- so each row below has to be read on its own scale. To judge
whether a value is meaningfully non-zero we need a reference, so for each SPI we compute
a **null band**: the largest magnitude it produces on pairs we know are unrelated.
Anything at or below that is indistinguishable from noise at this sample size.

In [ ]:
UNDIRECTED = ["cov_EmpiricalCovariance", "spearmanr", "dcorr", "mi_kraskov_NN-4"]
SHOWN = [("drive", "lin"), ("drive", "nonlin"), ("drive", "indep")]

# Every pair with no contemporaneous dependence. src/tgt belongs here: their coupling is
# purely at lag 1, so a contemporaneous statistic should see nothing.
COUPLED = {("drive", "lin"), ("drive", "nonlin"), ("lin", "nonlin")}
NULL_PAIRS = [p for p in combinations(NAMES, 2) if p not in COUPLED]

comparison = pd.DataFrame({
    spi: {**{f"{a}-{b}": abs(matrix(spi).loc[a, b]) for a, b in SHOWN},
          "null band": max(abs(matrix(spi).loc[a, b]) for a, b in NULL_PAIRS)}
    for spi in UNDIRECTED
}).T
comparison.round(3)

**Read the `drive-nonlin` column against `null band`.** Covariance and Spearman put the
nonlinear pair at or below their own null band: on their evidence the two processes are
unrelated. Distance correlation and KSG mutual information both put it well clear.

Mutual information ranks `drive-nonlin` *above* `drive-lin`, which is correct rather than
surprising: `nonlin` is a deterministic function of `drive` plus a small noise term,
whereas `lin` carries a larger independent noise component, so `drive` tells you more
about `nonlin` than about `lin`.

Spearman is the honest caveat here. It is a rank statistic, so it is often described as
"nonlinear", but it only detects *monotonic* dependence -- and `x -> x**2` is not
monotonic on a symmetric domain. It fails on this example for the same reason covariance
does.

#### 4. Direction

`src` drives `tgt` at lag 1 and nothing flows back. Recall the orientation convention:
entry `[i, j]` treats `i` as source and `j` as target, so a directed SPI should be large
at `[src, tgt]` and near zero at `[tgt, src]`.

In [ ]:
for spi in ["tlmi_gaussian", "te_kraskov_NN-4_k-1_kt-1_l-1_lt-1", "dcorr"]:
    m = matrix(spi)
    print(f"{spi}\n    src -> tgt : {m.loc['src', 'tgt']: .3f}"
          f"\n    tgt -> src : {m.loc['tgt', 'src']: .3f}")

In [ ]:
PANELS = [("dcorr", "Distance correlation\n(undirected)"),
          ("tlmi_gaussian", "Time-lagged MI\n(directed)"),
          ("te_kraskov_NN-4_k-1_kt-1_l-1_lt-1", "Transfer entropy\n(directed)")]

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.9))

for ax, (spi, title) in zip(axes, PANELS):
    v = matrix(spi).values
    im = ax.imshow(v, cmap="Blues", vmin=0, vmax=np.nanmax(v))

    ax.set_xticks(range(len(NAMES)), NAMES, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(NAMES)), NAMES, fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("target", fontsize=9)
    ax.spines[:].set_visible(False)

    # Mark the true coupling and its reverse.
    i, j = NAMES.index("src"), NAMES.index("tgt")
    for (r, c), colour in (((i, j), "#c1440e"), ((j, i), "#7a7a7a")):
        ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1, fill=False,
                                   edgecolor=colour, lw=2))
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

axes[0].set_ylabel("source", fontsize=9)
fig.suptitle("Orange = true coupling src -> tgt;  grey = the reverse", fontsize=10)
fig.tight_layout()
plt.show()

The undirected panel is symmetric by construction, and because distance correlation is
contemporaneous it registers essentially nothing for `src`/`tgt` at all -- the coupling
lives entirely at lag 1. Both directed SPIs light up a single off-diagonal cell in the
correct orientation and leave the reverse at the noise floor.

Note that "directed" and "lagged" are separate axes. `dcorr` fails here for two
independent reasons: it is undirected *and* contemporaneous. A lagged undirected
statistic (e.g. `CrossCorrelation`) would find the coupling but not its direction.

#### 5. Caveats

- **Sample size.** The KSG estimators have a visible noise floor -- their values on
  genuinely independent pairs are small but not zero, and can be slightly negative. That
  is why the comparison above is made against a measured null band rather than against
  zero. At `T = 200` the nonlinear detection still works but the margin narrows
  considerably; below that, treat single-pair values with suspicion.
- **One realisation, one seed.** The qualitative pattern is a structural consequence of
  the estimators and does not depend on the seed. The specific numbers do.
- **Not a significance test.** The null band is a crude within-dataset reference, not a
  p-value. For inference, use a surrogate or permutation null.
- **No statistic is best.** The point is not that mutual information beats correlation.
  Covariance is far cheaper, has a known sampling distribution, and is the right choice
  when the relationship really is linear. The point is that committing to one statistic
  commits you to its blind spots, and pyspi makes it cheap to not commit.

##### Next

`pyspi.utils.filter_spis` builds keyword-filtered configs from the SPI labels
(`directed`, `undirected`, `linear`, `nonlinear`, `signed`, `unsigned`,
`contemporaneous`, `time-dependent`, `frequency-dependent`), which is the usual way to go
from these six SPIs to a broader but still affordable set.